# BC카드 공모전 — H1(선행성) 시차상관 통계 검증

카드 초과성장률(전국 대비)과 외국인 방문자수(한국관광 데이터랩) 증가율 사이의 시차상관(lag correlation)을 계산한다. Claude가 로컬에서 먼저 돌려본 결과를 그대로 재현할 수 있도록, 드라이브에 있는 실제 파일(`bc_preprocessed.pkl`, `tour_datalab_foreign_visitors_monthly.csv`)에서 직접 수치를 읽어온다 — 숫자를 손으로 옮겨 적지 않아서 오타 위험이 없다.

**참고**: 이전 대화에서 언급된 "393%"는 청송군의 3월→4월 외국인 방문자수 증가율이다(2,537명 → 12,514명). 이 시기는 청송군 사과농가 계절근로자가 4/27부터 순차 입국한 시점과 겹친다 — 즉 관광객 급증이 아니라 계절근로자 입국이 만든 스파이크일 가능성이 높다는 게 이미 확인된 부분.

## 0. 환경 설정

In [ ]:
import pickle
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/BC카드'


## 1. 카드 데이터 로드 (1단계 전처리 결과)
`bc_preprocessed.pkl`의 `pivot_cnt`는 이미 외국인(GENDER_CD=3) 세그먼트만 필터링된 데이터다 — `national_cnt` 총합(7,550만 건)이 데이터 사전에 적어둔 외국인 세그먼트 총 CNT와 정확히 일치하는 것으로 확인됨. 즉 아래 카드 초과성장은 처음부터 외국인 기준이라, TourAPI의 외국인 방문자수(touDivCd=3)와 세그먼트가 그대로 맞아떨어진다.
**필터 기준(수정)**: 지역별 카드 건수는 1단계의 최소건수 필터(누적 ≥200건 · 최근월 ≥30건)를 통과한 `pivot_cnt_reliable`로 합산한다. 전국 기준선(`national_cnt`)은 필터 없이 전체를 쓴다.

In [ ]:
with open(f'{BASE_DIR}/bc_preprocessed.pkl', 'rb') as f:
    prep = pickle.load(f)

# 최소건수 필터(누적 ≥200 · 최근월 ≥30)를 통과한 지역×업종만 합산한다 — 제출 자료의 수치(ρ=-0.591)와 같은 기준.
# False로 바꾸면 필터를 적용하지 않은 합계 → ρ=-0.555 (p=0.001)로 결론(부호·유의성)은 같다.
USE_RELIABLE = True
pivot_cnt = prep['pivot_cnt_reliable'] if USE_RELIABLE else prep['pivot_cnt']   # index: (SIDO_NM, CCG_NM, TP_BUZ_NM_CLEAN), columns: 202601~202606
national_cnt = prep['national_cnt']  # 전국 기준선은 항상 필터 미적용 전체 (index: 업종 11개, columns: 202601~202606)
MONTHS = list(pivot_cnt.columns)

national_monthly = national_cnt.sum(axis=0)  # 전국 월별 총 CNT (외국인 기준)
print('전국 월별 CNT(외국인):', national_monthly.tolist())
print('6개월 총합:', int(national_monthly.sum()), '건 (데이터 사전의 7,550만 건과 일치하는지 확인)')


## 2. 방문자 데이터 로드 (5단계 한국관광 데이터랩)

In [ ]:
visitor_df = pd.read_csv(f'{BASE_DIR}/tour_datalab_foreign_visitors_monthly.csv')
visitor_df = visitor_df.set_index('STRD_YYMM')
visitor_df.index = visitor_df.index.astype(str)
print(visitor_df)


## 3. 후보지 정의 및 카드 CNT 월별 합계 추출

In [ ]:
# (SIDO_NM, CCG_NM) → TourAPI 쪽 지역명(visitor_df 컬럼명)
TARGET_REGIONS = {
    ('경상북도', '청송군'): '청송군',
    ('경상북도', '봉화군'): '봉화군',
    ('경상북도', '영양군'): '영양군',
    ('경상북도', '의성군'): '의성군',
    ('강원특별자치도', '고성군'): '고성군',
    ('강원특별자치도', '영월군'): '영월군',
    ('강원특별자치도', '양양군'): '양양군',
    ('인천광역시', '옹진군'): '옹진군',
}

card_monthly = {}
for (sido, ccg), name in TARGET_REGIONS.items():
    sub = pivot_cnt.loc[(sido, ccg)]  # 업종 11개 x 6개월
    card_monthly[name] = sub.sum(axis=0)  # 업종 합산 → 월별 총 CNT

card_monthly_df = pd.DataFrame(card_monthly).T
card_monthly_df.columns = MONTHS
print(card_monthly_df)


## 4. MoM% 및 카드 초과성장률 계산

In [ ]:
def mom_pct(series):
    vals = list(series)
    return [ (vals[i]/vals[i-1] - 1) * 100 for i in range(1, len(vals)) ]  # 길이 5, 2~6월

nat_mom = mom_pct(national_monthly)
print('전국 MoM%:', [round(x,1) for x in nat_mom])

card_excess = {}
visitor_mom = {}
for name in TARGET_REGIONS.values():
    c_mom = mom_pct(card_monthly_df.loc[name])
    card_excess[name] = [c - n for c, n in zip(c_mom, nat_mom)]
    visitor_mom[name] = mom_pct(visitor_df[name])

summary = pd.DataFrame({
    '카드 초과성장(2~6월)': {k: [round(x,1) for x in v] for k,v in card_excess.items()},
    '방문자 MoM%(2~6월)': {k: [round(x,1) for x in v] for k,v in visitor_mom.items()},
})
summary


## 5. 시차상관(lag correlation) 분석
8개 지역 × 5개월(2~6월) 데이터를 모아서 세 가지 시차로 상관을 본다:
- **lag 0**: 같은 달 카드 vs 방문자
- **lag +1**: 방문자[이번달] → 카드[다음달] (방문자 선행 가설)
- **lag -1**: 카드[이번달] → 방문자[다음달] (카드 선행 가설 = 원래 H1)

극단치(예: 청송군 393%)가 있어서 Pearson(선형)과 함께 이상치에 강한 Spearman(순위)도 같이 본다.

In [ ]:
def build_lag_pairs(card_excess, visitor_mom, lag):
    xs, ys = [], []
    for name in card_excess:
        ce = card_excess[name]  # idx0=2월 ... idx4=6월
        vm = visitor_mom[name]
        if lag == 0:
            for i in range(5):
                xs.append(vm[i]); ys.append(ce[i])
        elif lag == 1:  # 방문자[i] vs 카드[i+1]
            for i in range(4):
                xs.append(vm[i]); ys.append(ce[i+1])
        elif lag == -1:  # 카드[i] vs 방문자[i+1]
            for i in range(4):
                xs.append(ce[i]); ys.append(vm[i+1])
    return xs, ys

labels = {0: '동시(lag 0)', 1: '방문자→카드 (lag +1, 방문자 선행 가설)', -1: '카드→방문자 (lag -1, 카드 선행 가설=H1)'}
for lag in [0, 1, -1]:
    xs, ys = build_lag_pairs(card_excess, visitor_mom, lag)
    r_p, p_p = pearsonr(xs, ys)
    r_s, p_s = spearmanr(xs, ys)
    print(f"[{labels[lag]}] n={len(xs)} | Pearson r={r_p:.3f} (p={p_p:.4f}) | Spearman rho={r_s:.3f} (p={p_s:.4f})")


## 6. 강건성 재검증 — 지역별 평균 제거(demean) 후 lag -1 재계산
n=32는 8개 지역 × 4개월의 반복측정(패널데이터)이라 완전한 독립표본이 아니다. 지역마다 원래 변동폭이 다른 게 상관을 부풀렸을 수 있어서, 지역별 평균을 빼고 편차만으로 다시 검증한다.

In [ ]:
def demean(d):
    return {k: (np.array(v) - np.mean(v)).tolist() for k, v in d.items()}

ce_dm = demean(card_excess)
vm_dm = demean(visitor_mom)
xs, ys = build_lag_pairs(ce_dm, vm_dm, -1)
r_p, p_p = pearsonr(xs, ys)
r_s, p_s = spearmanr(xs, ys)
print(f"[지역평균 제거 후 lag -1] n={len(xs)} | Pearson r={r_p:.3f} (p={p_p:.4f}) | Spearman rho={r_s:.3f} (p={p_s:.4f})")
print()
print('해석: 지역별 평균수준 차이를 없애도 유의성이 유지되면, 이건 지역 간 스케일 차이로 인한 착시가 아니라')
print('진짜 각 지역 안에서 방문자 증가 사이클이 카드 증가 사이클보다 먼저 온다는 패턴이라는 뜻.')


## 7. (선택) 시각화 — 청송군 예시로 두 시계열의 산 모양 확인

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(8,4))
x = range(2, 7)
ax1.plot(x, card_excess['청송군'], 'o-', color='tab:blue', label='카드 초과성장(%p)')
ax1.set_ylabel('카드 초과성장 (%p)', color='tab:blue')
ax2 = ax1.twinx()
ax2.plot(x, visitor_mom['청송군'], 's--', color='tab:red', label='방문자 MoM(%)')
ax2.set_ylabel('외국인 방문자 MoM (%)', color='tab:red')
ax1.set_xlabel('월')
ax1.set_xticks(list(x))
plt.title('청송군: 카드 초과성장 vs 외국인 방문자 증가율')
fig.tight_layout()
plt.show()
